In [ ]:
"""
Multi-task training on MSI tiles using df_meta.csv

Assumptions:
- df_meta.csv has columns:
    sample_path, dataset_id, tile_r, tile_c, tile_h, tile_w, channels,
    organism, polarity, Organism_Part, Condition, analyzerType,
    ionisationSource, split
- sample_path points to a .npy file containing a single tile as a
  NumPy array of shape (C, H, W), where C = channels column.
- 'split' is one of: 'train', 'val', 'test'.

This script:
- Loads tiles from .npy
- Builds a multi-task ViT model (Dinov2-ViT-B/14 backbone)
- Trains heads only (Phase 1), then fine-tunes last N transformer blocks (Phase 2)
- Uses multi-task cross-entropy + optional supervised contrastive loss
  on the organism labels to promote clusterability (higher ARI).

NEW:
- Phase 1 can be skipped if already done: set cfg.phase1_epochs = 0 and
  ensure cfg.phase1_ckpt_path points to a trained Phase 1 checkpoint.
- No per-batch "loss=..." lines; loss is shown in-place on the tqdm bar.
"""

import os
import random
from dataclasses import dataclass
from typing import Dict, List, Optional
from tqdm import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

# ============================================================
# CONFIG
# ============================================================
SEED = 6740

@dataclass
class TrainConfig:
    df_meta_path: str = "df_meta.csv"

    # If sample_path is relative, prepend this base directory
    base_dir: str = ""  # e.g. "metaspace_images_dump"

    # Tasks (must match df columns)
    tasks: List[str] = None  # set below
    ignore_index: int = -100

    # Data loading
    batch_size: int = 64
    num_workers: int = 0

    # Model / training phases
    img_size: int = 224           # tile_h/tile_w in df_meta (you can crop later if you want)
    backbone_name: str = "vit_base_patch14_dinov2.lvd142m" #deit_base_distilled_patch16_224, vit_base_patch14_dinov2.lvd142m, vit_base_patch16_224.mae

    # Phase 1: heads only
    phase1_epochs: int = 8
    phase1_head_lr: float = 1e-3

    # Phase 2: fine-tune backbone
    phase2_epochs: int = 60
    phase2_head_lr: float = 1e-3
    phase2_backbone_lr: float = 1e-5
    unfreeze_last_n_blocks: int = 4

    # Checkpoint paths
    phase1_ckpt_path: str = os.path.join("checkpoints", "multitask_dinov2_phase1.pt")
    final_ckpt_path: str = os.path.join("checkpoints", "multitask_dinov2_final.pt")

    weight_decay: float = 0.01
    use_amp: bool = True
    early_stop_patience: int = 10

    # Multi-task loss weights
    task_weights: Dict[str, float] = None  # set below

    # SupCon on organism embeddings
    supcon_weight: float = 0.0
    supcon_temperature: float = 0.2


cfg = TrainConfig()
if cfg.tasks is None:
    cfg.tasks = [
        "organism",
        "polarity",
        "Organism_Part",
        "Condition",
        "analyzerType",
        "ionisationSource",
    ]

if cfg.task_weights is None:
    cfg.task_weights = {
        "organism": 1.0,
        "Organism_Part": 1.0,
        "polarity": 0.8,
        "analyzerType": 0.5,
        "ionisationSource": 0.5,
        "Condition": 0.3,
    }

# ============================================================
# UTILS / SEEDING
# ============================================================
def set_seed(seed: int = 6740):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# ============================================================
# DATASET
# ============================================================
class MSITilesFromNpy(Dataset):
    """
    Dataset that reads tiles from .npy or .npz files using df_meta.csv rows.

    Standardizes all tiles to (target_channels, target_h, target_w)
    so that DataLoader can stack them into batches.

    Expects df to contain columns:
        sample_path, channels, and label columns (tasks).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        base_dir: str,
        task_cols: List[str],
        task_label_maps: Optional[Dict[str, Dict[str, int]]] = None,
        ignore_index: int = -100,
        target_channels: Optional[int] = None,
        target_h: Optional[int] = None,
        target_w: Optional[int] = None,
    ):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir
        self.task_cols = task_cols
        self.ignore_index = ignore_index

        # Target shapes (global)
        self.target_channels = target_channels
        self.target_h = target_h
        self.target_w = target_w

        # Build label maps if not provided
        self.task_label_maps = task_label_maps or {}
        for t in self.task_cols:
            if t not in self.df.columns:
                print(f"[WARN] Task column '{t}' not in df_meta; will be ignored.")
                self.task_label_maps[t] = {}
                continue

            if t not in self.task_label_maps:
                col = self.df[t].astype("string")
                uniques_raw = col.unique()
                cleaned = []
                for u in uniques_raw:
                    if pd.isna(u):
                        continue
                    s = str(u)
                    if s in ["<NA>", "nan", "NaN", "None"]:
                        continue
                    cleaned.append(s)
                uniques = sorted(set(cleaned))
                label_map = {cls: i for i, cls in enumerate(uniques)}
                self.task_label_maps[t] = label_map
                print(f"[INFO] Task '{t}' has {len(label_map)} classes.")

        # Just for logging
        self.channels_unique = sorted(self.df["channels"].unique())
        print(f"[INFO] Unique channel counts in this split: {self.channels_unique}")
        if len(self.channels_unique) > 1:
            print("[WARN] Multiple channel counts present; they will be padded/trimmed to "
                  f"C={self.target_channels}.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        sample_path = row["sample_path"]

        if self.base_dir and not os.path.isabs(sample_path):
            path = os.path.join(self.base_dir, sample_path)
        else:
            path = sample_path

        # ---- load npy / npz robustly ----
        loaded = np.load(path)
        if isinstance(loaded, np.lib.npyio.NpzFile):
            if "arr_0" in loaded.files:
                arr = loaded["arr_0"]
            else:
                arr = loaded[loaded.files[0]]
        else:
            arr = loaded

        arr = np.asarray(arr)

        # ---- sanitize NaNs / infs ----
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

        # optional: simple per-tile scaling to [0, 1] to avoid huge magnitudes
        vmax = arr.max()
        if vmax > 0:
            arr = arr / vmax

        # Sometimes there is an extra singleton dim, squeeze it
        if arr.ndim > 3:
            arr = np.squeeze(arr)

        if arr.ndim != 3:
            raise ValueError(f"Expected (C,H,W) or (H,W,C) array, got shape {arr.shape} at {path}")

        expected_c = int(row["channels"])

        # Make sure channels are the first dim -> (C,H,W)
        if arr.shape[0] == expected_c:
            # already (C,H,W)
            pass
        elif arr.shape[-1] == expected_c:
            # probably (H,W,C) -> transpose
            arr = np.moveaxis(arr, -1, 0)
        else:
            # best-effort: assume first dim is channels
            pass

        C, H, W = arr.shape

        # ---- channel padding/truncation to target_channels ----
        if self.target_channels is not None:
            C_target = self.target_channels
            if C < C_target:
                pad = np.zeros((C_target - C, H, W), dtype=arr.dtype)
                arr = np.concatenate([arr, pad], axis=0)
            elif C > C_target:
                arr = arr[:C_target, :, :]
            C, H, W = arr.shape

        x = torch.from_numpy(arr).float()  # (C,H,W)

        # ---- spatial resize to (target_h, target_w) ----
        if self.target_h is not None and self.target_w is not None:
            x = x.unsqueeze(0)  # (1,C,H,W)
            x = F.interpolate(
                x,
                size=(self.target_h, self.target_w),
                mode="bilinear",
                align_corners=False,
            )
            x = x.squeeze(0)  # (C,H,W)

        labels = {}
        for t in self.task_cols:
            if t not in self.df.columns:
                labels[t] = self.ignore_index
                continue

            val = row[t]
            if pd.isna(val):
                labels[t] = self.ignore_index
            else:
                s = str(val)
                lm = self.task_label_maps.get(t, {})
                labels[t] = lm.get(s, self.ignore_index)

        return x, labels

# ============================================================
# MODEL: BACKBONE + MULTI-TASK HEADS
# ============================================================
class MultiTaskViT(nn.Module):
    def __init__(
        self,
        backbone_name: str,
        in_chans: int,
        task_label_maps: Dict[str, Dict[str, int]],
        img_size: int,
    ):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0,   # return embedding
            in_chans=in_chans,
            img_size=img_size,
        )

        # disable strict size check if present
        if hasattr(self.backbone, "patch_embed") and hasattr(self.backbone.patch_embed, "strict_img_size"):
            self.backbone.patch_embed.strict_img_size = False

        embed_dim = self.backbone.num_features

        self.tasks = list(task_label_maps.keys())
        self.heads = nn.ModuleDict()
        for t, lm in task_label_maps.items():
            num_classes = len(lm)
            if num_classes <= 1:
                num_classes = max(num_classes, 1)
            self.heads[t] = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        z = self.backbone(x)  # (B, D)
        logits = {t: head(z) for t, head in self.heads.items()}
        return z, logits


# ============================================================
# LOSSES
# ============================================================
def compute_multitask_ce_loss(
    logits: Dict[str, torch.Tensor],
    labels: Dict[str, torch.Tensor],
    task_weights: Dict[str, float],
    ignore_index: int,
):
    total_loss = 0.0
    task_losses = {}

    for t, logit in logits.items():
        if t not in labels:
            continue
        y = labels[t]
        if y is None:
            continue

        # mask out ignore_index
        valid_mask = y.ne(ignore_index)
        if not valid_mask.any():
            # no valid labels for this task in this batch -> skip
            continue

        logit_valid = logit[valid_mask]
        y_valid = y[valid_mask]

        loss = F.cross_entropy(
            logit_valid,
            y_valid,
            reduction="mean",
        )
        w = task_weights.get(t, 1.0)
        total_loss = total_loss + w * loss
        task_losses[t] = loss.detach().item()

    # if no tasks contributed, force zero loss (avoid NaN)
    if len(task_losses) == 0:
        total_loss = torch.tensor(0.0, device=next(iter(logits.values())).device)

    return total_loss, task_losses

def supervised_contrastive_loss(
    z: torch.Tensor,
    labels: torch.Tensor,
    temperature: float = 0.2,
    ignore_index: int = -100,
):
    """
    Simple supervised contrastive loss on embeddings z using labels.

    z: (N, D)
    labels: (N,)
    """
    valid_mask = labels.ne(ignore_index)
    z = z[valid_mask]
    labels = labels[valid_mask]

    if z.size(0) < 2:
        return torch.tensor(0.0, device=z.device)

    z = F.normalize(z, dim=-1)
    sim = torch.matmul(z, z.T) / temperature  # (N,N)

    # mask self
    logits_mask = torch.eye(sim.size(0), device=sim.device, dtype=torch.bool)
    sim = sim.masked_fill(logits_mask, float("-inf"))

    labels = labels.view(-1, 1)
    matches = torch.eq(labels, labels.T).float()
    pos_mask = matches * (~logits_mask).float()

    log_prob = F.log_softmax(sim, dim=1)
    pos_log_prob = (log_prob * pos_mask).sum(dim=1) / (pos_mask.sum(dim=1) + 1e-6)

    loss = -pos_log_prob.mean()
    return loss


# ============================================================
# TRAINING HELPERS
# ============================================================
def freeze_backbone(model: MultiTaskViT, freeze: bool = True):
    for p in model.backbone.parameters():
        p.requires_grad = not freeze


def unfreeze_last_n_blocks(model: MultiTaskViT, n: int):
    """
    For timm ViT: model.backbone.blocks is a ModuleList of transformer blocks.
    We unfreeze only the last n blocks; earlier ones stay frozen.
    """
    freeze_backbone(model, freeze=True)
    if not hasattr(model.backbone, "blocks"):
        print("[WARN] Backbone has no 'blocks' attribute; unfreezing entire backbone.")
        freeze_backbone(model, freeze=False)
        return

    blocks = model.backbone.blocks
    for blk in blocks[-n:]:
        for p in blk.parameters():
            p.requires_grad = True


def build_optimizers(model: MultiTaskViT, cfg: TrainConfig, phase: int):
    params_backbone = []
    params_heads = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("backbone."):
            params_backbone.append(param)
        else:
            params_heads.append(param)

    if phase == 1:
        optim = torch.optim.AdamW(
            params_heads,
            lr=cfg.phase1_head_lr,
            weight_decay=cfg.weight_decay,
        )
        scheduler = None
    else:
        optim = torch.optim.AdamW(
            [
                {"params": params_backbone, "lr": cfg.phase2_backbone_lr},
                {"params": params_heads, "lr": cfg.phase2_head_lr},
            ],
            weight_decay=cfg.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optim, T_max=max(cfg.phase2_epochs, 1)
        )
    return optim, scheduler


def eval_on_loader(model, loader, cfg: TrainConfig):
    model.eval()
    ce_losses = []

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device, non_blocking=True)
            labels_torch = {t: labels[t].to(device, non_blocking=True)
                            for t in labels}

            z, logits = model(x)
            loss_ce, _ = compute_multitask_ce_loss(
                logits, labels_torch, cfg.task_weights, cfg.ignore_index
            )
            ce_losses.append(loss_ce.item())

    return float(np.mean(ce_losses)) if ce_losses else float("inf")


# ============================================================
# DATA LOADERS FROM DF_META
# ============================================================
def build_loaders_from_dfmeta(cfg: TrainConfig):
    df = pd.read_csv(cfg.df_meta_path, sep=None, engine="python")

    required_cols = {"sample_path", "channels", "split"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"df_meta.csv missing columns: {missing}")

    # ============================================================
    # GLOBAL FILTER: drop classes with < MIN_CLASS_COUNT over ALL data
    # ============================================================
    MIN_CLASS_COUNT = 100
    keep_classes_by_task = {}

    for t in cfg.tasks:
        if t not in df.columns:
            print(f"[WARN] Task '{t}' not in df_meta; skipping in MIN_CLASS_COUNT filtering.")
            continue

        # Count only non-NA labels
        vc = df[t].dropna().value_counts()
        keep = vc[vc >= MIN_CLASS_COUNT].index.tolist()
        keep_classes_by_task[t] = set(keep)

        print(f"[INFO] Task '{t}': keeping {len(keep)} classes with ≥ {MIN_CLASS_COUNT} samples.")

    # Build a global mask: keep rows where every task label is either
    #  - NA (will become ignore_index) OR
    #  - in the kept class set for that task
    mask = np.ones(len(df), dtype=bool)
    for t, keep_set in keep_classes_by_task.items():
        if t not in df.columns or len(keep_set) == 0:
            # If no kept classes for this task, we don't enforce filtering on it
            continue
        col = df[t]
        mask &= (col.isna() | col.isin(keep_set))

    before_rows = len(df)
    df = df[mask].reset_index(drop=True)
    after_rows = len(df)
    print(f"[INFO] Global MIN_CLASS_COUNT filtering: {before_rows - after_rows} rows removed, {after_rows} remain.")

    # Optional: warn if any task ended up with <2 classes
    for t in cfg.tasks:
        if t in df.columns:
            n_classes = df[t].dropna().nunique()
            if n_classes < 2:
                print(f"[WARN] Task '{t}' has only {n_classes} classes after filtering. "
                      f"Consider removing this task from cfg.tasks.")

    # ============================================================
    # Now split by 'split' AFTER filtering
    # ============================================================
    df_train = df[df["split"] == "train"].reset_index(drop=True)
    df_val   = df[df["split"] == "val"].reset_index(drop=True) \
               if "val" in df["split"].unique() else pd.DataFrame(columns=df.columns)
    df_test  = df[df["split"] == "test"].reset_index(drop=True) \
               if "test" in df["split"].unique() else pd.DataFrame(columns=df.columns)

    print(f"[INFO] df_meta rows after filtering: train={len(df_train)} val={len(df_val)} test={len(df_test)}")

    # Determine global target channels and spatial size
    global_in_chans = int(df["channels"].max())
    target_h = cfg.img_size
    target_w = cfg.img_size
    print(f"[INFO] Using target shape: C={global_in_chans}, H={target_h}, W={target_w}")

    # Build train dataset (constructs label maps)
    train_ds = MSITilesFromNpy(
        df=df_train,
        base_dir=cfg.base_dir,
        task_cols=cfg.tasks,
        task_label_maps=None,
        ignore_index=cfg.ignore_index,
        target_channels=global_in_chans,
        target_h=target_h,
        target_w=target_w,
    )
    label_maps = train_ds.task_label_maps

    # Helper to build val/test with same label maps & target shapes
    def make_ds(sub_df):
        if len(sub_df) == 0:
            return None
        return MSITilesFromNpy(
            df=sub_df,
            base_dir=cfg.base_dir,
            task_cols=cfg.tasks,
            task_label_maps=label_maps,
            ignore_index=cfg.ignore_index,
            target_channels=global_in_chans,
            target_h=target_h,
            target_w=target_w,
        )

    val_ds = make_ds(df_val)
    test_ds = make_ds(df_test)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    ) if val_ds is not None else None
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    ) if test_ds is not None else None

    return train_loader, val_loader, test_loader, label_maps, global_in_chans

# ============================================================
# MAIN TRAIN LOOP
# ============================================================
def train(cfg: TrainConfig):
    # Ensure checkpoint directory exists
    if cfg.phase1_ckpt_path:
        os.makedirs(os.path.dirname(cfg.phase1_ckpt_path), exist_ok=True)
    if cfg.final_ckpt_path:
        os.makedirs(os.path.dirname(cfg.final_ckpt_path), exist_ok=True)

    train_loader, val_loader, test_loader, label_maps, in_chans = build_loaders_from_dfmeta(cfg)

    model = MultiTaskViT(
        backbone_name=cfg.backbone_name,
        in_chans=in_chans,
        task_label_maps=label_maps,
        img_size=cfg.img_size,
    ).to(device)

    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

    # -----------------------------
    # PHASE 1: HEADS ONLY (OPTIONAL)
    # -----------------------------
    best_state_phase1 = None

    if cfg.phase1_epochs > 0:
        print("\n[PHASE 1] Training classification heads only...")
        freeze_backbone(model, freeze=True)
        optim, _ = build_optimizers(model, cfg, phase=1)

        best_val_loss = float("inf")

        for epoch in range(1, cfg.phase1_epochs + 1):
            model.train()
            running_loss = 0.0

            pbar = tqdm(train_loader, desc=f"[Phase1][Epoch {epoch}/{cfg.phase1_epochs}] Training", leave=False)
            for x, labels in pbar:
                x = x.to(device, non_blocking=True)
                labels_torch = {t: labels[t].to(device, non_blocking=True)
                                for t in labels}

                optim.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                    z, logits = model(x)
                    loss_ce, _ = compute_multitask_ce_loss(
                        logits, labels_torch, cfg.task_weights, cfg.ignore_index
                    )
                    loss = loss_ce

                if torch.isnan(loss) or torch.isinf(loss):
                    print("[WARN] NaN/Inf loss encountered in Phase 1; skipping batch.")
                    continue

                scaler.scale(loss).backward()
                scaler.step(optim)
                scaler.update()

                running_loss += loss.item() * x.size(0)
                pbar.set_postfix(loss=float(loss.item()))

            train_loss = running_loss / len(train_loader.dataset)
            if val_loader is not None:
                val_loss = eval_on_loader(model, val_loader, cfg)
            else:
                val_loss = train_loss

            print(f"[Phase1][Epoch {epoch}/{cfg.phase1_epochs}] "
                  f"train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_phase1 = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if best_state_phase1 is not None:
            model.load_state_dict(best_state_phase1)
            print("[Phase1] Loaded best model weights from Phase 1.")

            # Save a Phase 1-only checkpoint for future Phase 2 runs
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "label_maps": label_maps,
                    "cfg": cfg.__dict__,
                },
                cfg.phase1_ckpt_path,
            )
            print(f"[Phase1] Saved Phase 1 checkpoint to {cfg.phase1_ckpt_path}")

    elif cfg.phase2_epochs > 0:
        # No Phase 1 training requested; try to load pre-trained Phase 1 checkpoint
        if os.path.isfile(cfg.phase1_ckpt_path):
            ckpt = torch.load(cfg.phase1_ckpt_path, map_location="cpu")
            state_dict = ckpt.get("model_state", ckpt)
            model.load_state_dict(state_dict)
            print(f"[Phase1] Skipped training (phase1_epochs=0); "
                  f"loaded weights from {cfg.phase1_ckpt_path}")
        else:
            raise FileNotFoundError(
                f"[ERROR] phase1_epochs=0 but Phase 1 checkpoint not found at "
                f"{cfg.phase1_ckpt_path}. Run Phase 1 first or update cfg.phase1_ckpt_path."
            )
    else:
        print("[INFO] Phase 1 is disabled (phase1_epochs=0).")

    # -----------------------------
    # PHASE 2: FINE-TUNE BACKBONE (OPTIONAL)
    # -----------------------------
    best_state_phase2 = None

    if cfg.phase2_epochs > 0:
        print("\n[PHASE 2] Fine-tuning last backbone blocks + SupCon...")
        unfreeze_last_n_blocks(model, cfg.unfreeze_last_n_blocks)
        optim, scheduler = build_optimizers(model, cfg, phase=2)

        best_val_loss = float("inf")
        epochs_no_improve = 0

        for epoch in range(1, cfg.phase2_epochs + 1):
            model.train()
            running_loss = 0.0

            pbar = tqdm(train_loader, desc=f"[Phase2][Epoch {epoch}/{cfg.phase2_epochs}] Training", leave=False)
            for x, labels in pbar:
                x = x.to(device, non_blocking=True)
                labels_torch = {t: labels[t].to(device, non_blocking=True)
                                for t in labels}

                optim.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                    z, logits = model(x)
                    loss_ce, _ = compute_multitask_ce_loss(
                        logits, labels_torch, cfg.task_weights, cfg.ignore_index
                    )
                    loss = loss_ce

                    # Optional SupCon on organism
                    if cfg.supcon_weight > 0 and "organism" in labels_torch:
                        loss_supcon = supervised_contrastive_loss(
                            z,
                            labels_torch["organism"],
                            temperature=cfg.supcon_temperature,
                            ignore_index=cfg.ignore_index,
                        )
                        loss = loss + cfg.supcon_weight * loss_supcon

                if torch.isnan(loss) or torch.isinf(loss):
                    print("[WARN] NaN/Inf loss encountered in Phase 2; skipping batch.")
                    continue

                scaler.scale(loss).backward()
                scaler.step(optim)
                scaler.update()

                running_loss += loss.item() * x.size(0)
                pbar.set_postfix(loss=float(loss.item()))

            train_loss = running_loss / len(train_loader.dataset)
            if val_loader is not None:
                val_loss = eval_on_loader(model, val_loader, cfg)
            else:
                val_loss = train_loss

            if scheduler is not None:
                scheduler.step()

            print(f"[Phase2][Epoch {epoch}/{cfg.phase2_epochs}] "
                  f"train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_state_phase2 = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= cfg.early_stop_patience:
                    print("[Phase2] Early stopping triggered.")
                    break

        if best_state_phase2 is not None:
            model.load_state_dict(best_state_phase2)
            print("[Phase2] Loaded best model weights from Phase 2.")
    else:
        print("[INFO] Phase 2 is disabled (phase2_epochs=0); skipping fine-tuning.")

    # -----------------------------
    # FINAL TEST EVAL (CE only)
    # -----------------------------
    if test_loader is not None:
        test_loss = eval_on_loader(model, test_loader, cfg)
        print(f"[TEST] Final CE loss on test set: {test_loss:.4f}")

    # Save final checkpoint
    torch.save(
        {
            "model_state": model.state_dict(),
            "label_maps": label_maps,
            "cfg": cfg.__dict__,
        },
        cfg.final_ckpt_path,
    )
    print(f"[INFO] Saved model checkpoint to {cfg.final_ckpt_path}")

    return model, label_maps

In [ ]:
if __name__ == "__main__":
    train(cfg)

### Multi-task vs single task ablation

In [ ]:
import copy
import os
import numpy as np
import pandas as pd
import torch

# -------------------------------------------------------------------
# Helper: per-task accuracies (safe to redefine if already present)
# -------------------------------------------------------------------
def compute_task_accuracies(
    model: MultiTaskViT,
    loader,
    cfg: TrainConfig,
    label_maps,
):
    """
    Compute simple per-task accuracy on a given loader.
    Ignores samples with ignore_index.
    """
    model.eval()
    correct = {t: 0 for t in cfg.tasks}
    total   = {t: 0 for t in cfg.tasks}

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device, non_blocking=True)
            labels_torch = {t: labels[t].to(device, non_blocking=True)
                            for t in labels}

            _, logits = model(x)

            for t in cfg.tasks:
                if t not in logits or t not in labels_torch:
                    continue
                y = labels_torch[t]
                valid_mask = y.ne(cfg.ignore_index)
                if not valid_mask.any():
                    continue

                y_valid = y[valid_mask]
                pred = logits[t][valid_mask].argmax(dim=1)

                correct[t] += (pred == y_valid).sum().item()
                total[t]   += y_valid.numel()

    acc = {}
    for t in cfg.tasks:
        if total[t] > 0:
            acc[t] = correct[t] / total[t]
        else:
            acc[t] = float("nan")

    return acc


# -------------------------------------------------------------------
# Helper: generic wrapper around your existing train(cfg)
# -------------------------------------------------------------------
def _run_single_training_and_eval(cfg_local: TrainConfig, task_list):
    """
    Calls existing train(cfg_local) and ensures we always get test_metrics.

    Supports both:
      train(cfg) -> (model, label_maps)
      train(cfg) -> (model, label_maps, test_metrics)
    """
    cfg_local.tasks = task_list  # make sure tasks are set

    train_out = train(cfg_local)

    # Case 1: train returns (model, label_maps, test_metrics)
    if isinstance(train_out, (tuple, list)) and len(train_out) == 3:
        model, label_maps, test_metrics = train_out
        return model, label_maps, test_metrics

    # Case 2: train returns (model, label_maps)
    elif isinstance(train_out, (tuple, list)) and len(train_out) == 2:
        model, label_maps = train_out

        # Rebuild loaders to get test_loader
        train_loader, val_loader, test_loader, _, _ = build_loaders_from_dfmeta(cfg_local)

        test_metrics = {
            "test_ce_loss": float("nan"),
            "per_task_acc": {},
        }

        if test_loader is not None:
            test_loss = eval_on_loader(model, test_loader, cfg_local)
            test_accs = compute_task_accuracies(model, test_loader, cfg_local, label_maps)
            test_metrics["test_ce_loss"] = test_loss
            test_metrics["per_task_acc"] = test_accs

        return model, label_maps, test_metrics

    else:
        raise RuntimeError(
            f"train(cfg) returned unsupported object of type {type(train_out)} "
            f"with length {len(train_out) if isinstance(train_out, (tuple, list)) else 'N/A'}"
        )


# -------------------------------------------------------------------
# Helper: load your EXISTING multi-task phase1 ckpt and evaluate
# -------------------------------------------------------------------
def _eval_multitask_from_existing_phase1_ckpt(
    base_cfg: TrainConfig,
    task_list,
    ckpt_path: str,
):
    """
    Loads an existing multi-task Phase 1 checkpoint and evaluates it.
    This assumes the ckpt was saved from the same codebase (same backbone, img_size, etc.).
    """
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f"[ERROR] Multi-task phase1 checkpoint not found at: {ckpt_path}")

    print(f"\n[INFO] Loading existing multi-task Phase 1 checkpoint from:\n  {ckpt_path}\n")
    ckpt = torch.load(ckpt_path, map_location="cpu")

    # Get cfg info if stored; otherwise fall back to base_cfg
    ckpt_cfg_dict = ckpt.get("cfg", base_cfg.__dict__)
    backbone_name = ckpt_cfg_dict.get("backbone_name", base_cfg.backbone_name)
    img_size      = ckpt_cfg_dict.get("img_size", base_cfg.img_size)

    # Build loaders fresh to get label_maps and in_chans
    eval_cfg = copy.deepcopy(base_cfg)
    eval_cfg.tasks = task_list

    train_loader, val_loader, test_loader, label_maps, in_chans = build_loaders_from_dfmeta(eval_cfg)

    # Rebuild model with these label_maps & in_chans
    model = MultiTaskViT(
        backbone_name=backbone_name,
        in_chans=in_chans,
        task_label_maps=label_maps,
        img_size=img_size,
    ).to(device)

    state_dict = ckpt.get("model_state", ckpt)
    model.load_state_dict(state_dict)

    test_metrics = {
        "test_ce_loss": float("nan"),
        "per_task_acc": {},
    }

    if test_loader is not None:
        test_loss = eval_on_loader(model, test_loader, eval_cfg)
        test_accs = compute_task_accuracies(model, test_loader, eval_cfg, label_maps)

        test_metrics["test_ce_loss"] = test_loss
        test_metrics["per_task_acc"] = test_accs

        print(f"[MULTI] Loaded model test CE loss: {test_loss:.4f}")
        for t, acc in test_accs.items():
            if not np.isnan(acc):
                print(f"    [MULTI] {t}: acc = {acc:.4f}")
            else:
                print(f"    [MULTI] {t}: acc = NaN (no valid samples)")

    return model, label_maps, test_metrics


# -------------------------------------------------------------------
# FAST ABLATION: multi-task vs single-task (heads-only)
# -------------------------------------------------------------------
def run_ablation_multitask_vs_singletask_fast(
    base_cfg: TrainConfig,
    task_list=None,
    results_dir: str = "ablation_multitask_vs_singletask_fast",
    multitask_phase1_epochs: int = 8,
    singletask_phase1_epochs: int = 8,
):
    """
    FAST ABLATION:

      - Multi-task: **DO NOT retrain**; instead load the existing Phase 1
        checkpoint from:
          Y:\\coskun-lab\\Efe\\MSI Foundation Model\\ablation_multitask_vs_singletask_fast\\multitask_phase1.pt

      - Single-task: train fast heads-only models (Phase 1 only, Phase 2 disabled).

    Assumes all helpers from your main script are already defined.
    """
    os.makedirs(results_dir, exist_ok=True)

    if task_list is None:
        task_list = [
            "organism",
            "polarity",
            "Organism_Part",
            "Condition",
            "analyzerType",
            "ionisationSource",
        ]

    results = []

    # ============================================================
    # 1) MULTI-TASK RUN: LOAD EXISTING PHASE-1 CHECKPOINT
    # ============================================================
    multi_phase1_ckpt = r"Y:\coskun-lab\Efe\MSI Foundation Model\ablation_multitask_vs_singletask_fast\multitask_phase1.pt"

    print("\n================ USING EXISTING MULTI-TASK PHASE 1 CHECKPOINT ================\n")
    _, _, test_metrics_multi = _eval_multitask_from_existing_phase1_ckpt(
        base_cfg, task_list, multi_phase1_ckpt
    )

    for t in task_list:
        acc = test_metrics_multi["per_task_acc"].get(t, float("nan"))
        results.append({
            "setting": "multi_task",
            "task": t,
            "test_acc": acc,
            "test_ce_loss": test_metrics_multi["test_ce_loss"],
        })

    # ============================================================
    # 2) SINGLE-TASK RUNS (HEADS ONLY, FEWER EPOCHS)
    # ============================================================
    for t in task_list:
        cfg_single = copy.deepcopy(base_cfg)
        cfg_single.phase1_epochs = singletask_phase1_epochs
        cfg_single.phase2_epochs = 0  # disable backbone fine-tuning

        cfg_single.phase1_ckpt_path = os.path.join(results_dir, f"{t}_phase1.pt")
        cfg_single.final_ckpt_path  = os.path.join(results_dir, f"{t}_final.pt")

        print(f"\n================ FAST SINGLE-TASK RUN (HEADS ONLY): {t} ================\n")
        _, _, test_metrics_single = _run_single_training_and_eval(cfg_single, [t])

        acc_single = test_metrics_single["per_task_acc"].get(t, float("nan"))
        results.append({
            "setting": "single_task",
            "task": t,
            "test_acc": acc_single,
            "test_ce_loss": test_metrics_single["test_ce_loss"],
        })

    # ============================================================
    # 3) TO DATAFRAME + SAVE
    # ============================================================
    df_res = pd.DataFrame(results)
    df_pivot = df_res.pivot(index="task", columns="setting", values="test_acc").reset_index()
    df_pivot = df_pivot.rename_axis(None, axis=1)

    out_csv = os.path.join(results_dir, "ablation_multitask_vs_singletask_fast.csv")
    df_pivot.to_csv(out_csv, index=False)
    print(f"\n[FAST ABLATION] Saved ablation summary to {out_csv}")
    print(df_pivot)

    return df_pivot


if __name__ == "__main__":
    # This will:
    #   - Load your existing multi-task Phase 1 checkpoint from Y:\...
    #   - Train fast single-task heads-only models
    #   - Save a CSV with multi vs single accuracies
    run_ablation_multitask_vs_singletask_fast(cfg)

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
from collections import defaultdict
from sklearn.metrics import f1_score

# ==========================================================
# USER CONFIG
# ==========================================================
RESULTS_DIR = r"Y:\coskun-lab\Efe\MSI Foundation Model\ablation_multitask_vs_singletask_fast"

OUT_CSV = os.path.join(RESULTS_DIR, "ablation_eval_saved_checkpoints.csv")

# tasks to evaluate (must match df_meta + your training cfg)
TASK_LIST = [
    "organism",
    "polarity",
    "Organism_Part",
    "Condition",
    "analyzerType",
    "ionisationSource",
]

MULTITASK_CKPT = os.path.join(RESULTS_DIR, "multitask_phase1.pt")
SINGLE_CKPT_PATTERN = os.path.join(RESULTS_DIR, "{}_phase1.pt")  # {} replaced by task name


# ==========================================================
# METRIC FUNCTION (ACC + MACRO F1)
# ==========================================================
def compute_task_metrics(model, loader, cfg_eval):
    """
    Compute per-task test accuracy and macro F1.
    Uses cfg_eval.tasks and cfg_eval.ignore_index.
    """
    model.eval()
    correct = defaultdict(int)
    total   = defaultdict(int)
    y_true  = defaultdict(list)
    y_pred  = defaultdict(list)

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device, non_blocking=True)
            labels_torch = {t: labels[t].to(device, non_blocking=True)
                            for t in labels}

            _, logits = model(x)

            for t in cfg_eval.tasks:
                if t not in logits or t not in labels_torch:
                    continue
                y = labels_torch[t]
                valid_mask = y.ne(cfg_eval.ignore_index)
                if not valid_mask.any():
                    continue

                y_v = y[valid_mask]
                p_v = logits[t][valid_mask].argmax(dim=1)

                correct[t] += (p_v == y_v).sum().item()
                total[t]   += y_v.numel()

                y_true[t].extend(y_v.cpu().tolist())
                y_pred[t].extend(p_v.cpu().tolist())

    metrics = {"acc": {}, "macro_f1": {}}
    for t in cfg_eval.tasks:
        metrics["acc"][t] = correct[t] / total[t] if total[t] > 0 else float("nan")
        if len(y_true[t]) > 0:
            try:
                metrics["macro_f1"][t] = f1_score(y_true[t], y_pred[t], average="macro")
            except Exception:
                metrics["macro_f1"][t] = float("nan")
        else:
            metrics["macro_f1"][t] = float("nan")
    return metrics


# ==========================================================
# EVALUATION HELPER (NO TRAINING)
# ==========================================================
def evaluate_checkpoint(ckpt_path, base_cfg, task_subset):
    """
    Loads a checkpoint, rebuilds loaders and model, and computes:
      - CE loss on test set
      - per-task accuracy
      - per-task macro F1

    Assumes:
      - build_loaders_from_dfmeta, MultiTaskViT, eval_on_loader, device exist.
      - df_meta.csv is unchanged from training.
    """
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at: {ckpt_path}")

    print(f"[INFO] Evaluating checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cpu")

    ckpt_cfg = ckpt.get("cfg", base_cfg.__dict__)
    backbone_name = ckpt_cfg.get("backbone_name", base_cfg.backbone_name)
    img_size      = ckpt_cfg.get("img_size", base_cfg.img_size)

    # Use the existing training cfg (with task_weights, etc.) as base
    cfg_eval = copy.deepcopy(base_cfg)
    cfg_eval.tasks = task_subset

    # Build loaders and label maps from df_meta, same as training
    train_loader, val_loader, test_loader, label_maps, in_chans = build_loaders_from_dfmeta(cfg_eval)

    # Build model with these label maps, then load weights
    model = MultiTaskViT(
        backbone_name=backbone_name,
        in_chans=in_chans,
        task_label_maps=label_maps,
        img_size=img_size,
    ).to(device)

    state_dict = ckpt.get("model_state", ckpt)
    model.load_state_dict(state_dict)

    # CE loss
    test_loss = eval_on_loader(model, test_loader, cfg_eval)

    # Per-task metrics
    metrics = compute_task_metrics(model, test_loader, cfg_eval)

    return test_loss, metrics["acc"], metrics["macro_f1"]


# ==========================================================
# MAIN: EVAL MULTI-TASK + SINGLE-TASK
# ==========================================================
results = []

# Use your existing cfg from training as base (has task_weights, img_size, etc.)
base_cfg = cfg

# ---------------- MULTI-TASK ----------------
print("\n=== Evaluating MULTI-TASK checkpoint ===")
mt_loss, mt_acc, mt_f1 = evaluate_checkpoint(MULTITASK_CKPT, base_cfg, TASK_LIST)

for t in TASK_LIST:
    results.append({
        "setting": "multi_task",
        "task": t,
        "test_acc": mt_acc.get(t, float("nan")),
        "test_macro_f1": mt_f1.get(t, float("nan")),
        "test_ce_loss": mt_loss,
    })

# ---------------- SINGLE-TASK ----------------
for t in TASK_LIST:
    ckpt_path = SINGLE_CKPT_PATTERN.format(t)
    if not os.path.exists(ckpt_path):
        print(f"[WARN] Missing checkpoint for single-task: {t} ({ckpt_path})")
        continue

    print(f"\n=== Evaluating SINGLE-TASK checkpoint for: {t} ===")
    st_loss, st_acc, st_f1 = evaluate_checkpoint(ckpt_path, base_cfg, [t])

    results.append({
        "setting": "single_task",
        "task": t,
        "test_acc": st_acc.get(t, float("nan")),
        "test_macro_f1": st_f1.get(t, float("nan")),
        "test_ce_loss": st_loss,
    })

# ---------------- SAVE RESULTS ----------------
df_res = pd.DataFrame(results)
df_res.to_csv(OUT_CSV, index=False)
print("\nSaved evaluation results to:"
print(OUT_CSV)
print(df_res)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Paths
# -----------------------------
RESULTS_DIR = r"Y:\coskun-lab\Efe\MSI Foundation Model\ablation_multitask_vs_singletask_fast"
CSV_PATH    = os.path.join(RESULTS_DIR, "ablation_eval_saved_checkpoints.csv")

OUT_ACC_PNG = os.path.join(RESULTS_DIR, "ablation_acc_bars.png")
OUT_F1_PNG  = os.path.join(RESULTS_DIR, "ablation_macro_f1_bars.png")

# -----------------------------
# Load and check
# -----------------------------
df = pd.read_csv(CSV_PATH)

required_cols = {"task", "setting", "test_acc", "test_macro_f1"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in eval CSV: {missing}")

# Map setting names to nicer labels
setting_map = {
    "multi_task": "Multi-task",
    "single_task": "Single-task",
}
df["setting_pretty"] = df["setting"].map(setting_map)

# Sort tasks by multi-task accuracy for consistent ordering
multi_acc = (
    df[df["setting"] == "multi_task"]
    .set_index("task")["test_acc"]
    .fillna(-np.inf)
)
task_order = multi_acc.sort_values(ascending=False).index.tolist()
df["task"] = pd.Categorical(df["task"], categories=task_order, ordered=True)
df = df.sort_values("task")

# -----------------------------
# Matplotlib / Seaborn style
# -----------------------------
sns.set(style="whitegrid")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
})

palette = {
    "Multi-task": "#1f78b4",   # darker blue
    "Single-task": "#a6cee3",  # lighter blue
}

# ============================================================
# 1) Accuracy bar plot
# ============================================================
fig_acc, ax_acc = plt.subplots(figsize=(5.0, 3.0))

sns.barplot(
    data=df,
    x="task",
    y="test_acc",
    hue="setting_pretty",
    palette=palette,
    edgecolor="black",
    linewidth=0.4,
    ax=ax_acc,
)

ax_acc.set_ylabel("Test accuracy")
ax_acc.set_xlabel("Task")
ax_acc.set_ylim(0.0, 1.0)
ax_acc.set_title("Multi-task vs single-task (accuracy)", pad=4)

ax_acc.set_xticklabels(ax_acc.get_xticklabels(), rotation=30, ha="right")

ax_acc.spines["top"].set_visible(False)
ax_acc.spines["right"].set_visible(False)
ax_acc.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
ax_acc.legend(frameon=False, title="", loc="lower right")

plt.tight_layout()
fig_acc.savefig(OUT_ACC_PNG, dpi=600, bbox_inches="tight")
plt.close(fig_acc)

print(f"Saved accuracy bar plot to:\n  {OUT_ACC_PNG}")

# ============================================================
# 2) Macro F1 bar plot
# ============================================================
fig_f1, ax_f1 = plt.subplots(figsize=(5.0, 3.0))

sns.barplot(
    data=df,
    x="task",
    y="test_macro_f1",
    hue="setting_pretty",
    palette=palette,
    edgecolor="black",
    linewidth=0.4,
    ax=ax_f1,
)

ax_f1.set_ylabel("Macro F1 score")
ax_f1.set_xlabel("Task")
ax_f1.set_ylim(0.0, 1.0)
ax_f1.set_title("Multi-task vs single-task (macro F1)", pad=4)

ax_f1.set_xticklabels(ax_f1.get_xticklabels(), rotation=30, ha="right")

ax_f1.spines["top"].set_visible(False)
ax_f1.spines["right"].set_visible(False)
ax_f1.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
ax_f1.legend(frameon=False, title="", loc="lower right")

plt.tight_layout()
fig_f1.savefig(OUT_F1_PNG, dpi=600, bbox_inches="tight")
plt.close(fig_f1)

print(f"Saved macro F1 bar plot to:\n  {OUT_F1_PNG}")


In [ ]:
import os
import numpy as np
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
RESULTS_DIR = r"Y:\coskun-lab\Efe\MSI Foundation Model\ablation_multitask_vs_singletask_fast"
CSV_PATH    = os.path.join(RESULTS_DIR, "ablation_eval_saved_checkpoints.csv")
OUT_TEX     = os.path.join(RESULTS_DIR, "ablation_multitask_vs_singletask_table.tex")

# -----------------------------
# Load and sanity check
# -----------------------------
df = pd.read_csv(CSV_PATH)

required_cols = {"task", "setting", "test_acc", "test_macro_f1"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in eval CSV: {missing}")

# -----------------------------
# Pivot to wide format
#   index: task
#   columns: metric × setting
# -----------------------------
pivot = df.pivot_table(
    index="task",
    columns="setting",
    values=["test_acc", "test_macro_f1"],
    aggfunc="mean",  # should be single value per cell anyway
)

# Flatten multi-index columns
pivot.columns = [
    f"{metric}_{setting}"
    for metric, setting in pivot.columns.to_flat_index()
]

# Optional: sort tasks by multi-task accuracy desc for nice ordering
if "test_acc_multi_task" in pivot.columns:
    pivot = pivot.sort_values("test_acc_multi_task", ascending=False)

# -----------------------------
# Format numbers
# -----------------------------
def fmt(x):
    if pd.isna(x):
        return "-"
    return f"{x:.3f}"

pivot_formatted = pivot.applymap(fmt)

# Optionally rename columns to nicer LaTeX column headers
col_rename = {
    "test_acc_multi_task": r"Acc (Multi)",
    "test_acc_single_task": r"Acc (Single)",
    "test_macro_f1_multi_task": r"Macro F1 (Multi)",
    "test_macro_f1_single_task": r"Macro F1 (Single)",
}
pivot_formatted = pivot_formatted.rename(columns=col_rename)

# -----------------------------
# Build LaTeX table string
# -----------------------------
latex_table = pivot_formatted.to_latex(
    index=True,
    escape=False,              # so LaTeX in headers like "Acc (Multi)" stays fine
    column_format="lrrrr",     # 1 task column + 4 metric columns
    multicolumn=False,
)

# Wrap with full table environment
latex_output = (
    "\\begin{table}[ht]\n"
    "\\centering\n"
    "\\scriptsize\n"
    "\\setlength{\\tabcolsep}{6pt}\n"
    "\\renewcommand{\\arraystretch}{1.2}\n"
    "\\caption{Multi-task vs single-task performance across metadata prediction tasks. "
    "Values report test accuracy and macro F1.}\n"
    "\\label{tab:ablation_multitask_vs_singletask}\n"
    f"{latex_table}\n"
    "\\end{table}\n"
)

# -----------------------------
# Save to .tex
# -----------------------------
with open(OUT_TEX, "w") as f:
    f.write(latex_output)

print(f"Saved LaTeX ablation table to:\n  {OUT_TEX}")
print(latex_output)

### Export embeddings

In [ ]:
"""
Export embeddings from trained multitask Dinov2 model so they can be used
by the ARI + UMAP + pies script.

This will create:
    OUT_EMB_DIR/image_feats.npy-
    OUT_EMB_DIR/index.csv
which matches the format expected by your ARI script.
"""

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm  # <--- added

# -----------------------------
# Imports from your training script
# -----------------------------
# Make sure these are already defined in your environment:
# - MSITilesFromNpy
# - MultiTaskViT
# - TrainConfig, cfg
# - device

# -----------------------------
# Paths / config for export
# -----------------------------
DF_META_PATH = "df_meta.csv"  # same as in training
BASE_DIR = ""                 # same as cfg.base_dir if you used it
CKPT_PATH = "checkpoints/multitask_dinov2_final.pt"

OUT_EMB_DIR = os.path.join("pretrained_feats2", "msi_multitask_dinov2")
os.makedirs(OUT_EMB_DIR, exist_ok=True)

BATCH_SIZE = 64
NUM_WORKERS = 0

# -----------------------------
# Load df_meta and checkpoint
# -----------------------------
print("[EXPORT] Loading df_meta and checkpoint...")
df = pd.read_csv(DF_META_PATH, sep=None, engine="python").reset_index(drop=True)

ckpt = torch.load(CKPT_PATH, map_location=device)
label_maps = ckpt["label_maps"]
cfg_ckpt = ckpt["cfg"]

backbone_name = cfg_ckpt["backbone_name"]
img_size = cfg_ckpt["img_size"]
ignore_index = cfg_ckpt.get("ignore_index", -100)

# Determine global in_chans as in training
global_in_chans = int(df["channels"].max())
print(f"[EXPORT] Using in_chans={global_in_chans}, img_size={img_size}")

# -----------------------------
# Build dataset & loader (ALL rows, no split)
# -----------------------------
all_ds = MSITilesFromNpy(
    df=df,
    base_dir=BASE_DIR,
    task_cols=list(label_maps.keys()),
    task_label_maps=label_maps,       # reuse maps from training
    ignore_index=ignore_index,
    target_channels=global_in_chans,
    target_h=img_size,
    target_w=img_size,
)

all_loader = DataLoader(
    all_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# -----------------------------
# Rebuild model & load weights
# -----------------------------
model = MultiTaskViT(
    backbone_name=backbone_name,
    in_chans=global_in_chans,
    task_label_maps=label_maps,
    img_size=img_size,
).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

# -----------------------------
# Forward pass to collect embeddings (with tqdm)
# -----------------------------
print("[EXPORT] Running model to extract embeddings...")
all_embs = []
with torch.no_grad():
    for x, _labels in tqdm(all_loader, desc="[EXPORT] Extracting embeddings", leave=True):
        x = x.to(device, non_blocking=True)
        z, _ = model(x)          # z: (B, D) backbone embedding
        all_embs.append(z.cpu().numpy())

emb = np.concatenate(all_embs, axis=0)
assert emb.shape[0] == len(df), f"Embedding rows {emb.shape[0]} != df rows {len(df)}"
print(f"[EXPORT] Collected embeddings with shape: {emb.shape}")

# -----------------------------
# Save image_feats.npy + index.csv
# -----------------------------
feats_path = os.path.join(OUT_EMB_DIR, "image_feats.npy")
np.save(feats_path, emb)
print(f"[EXPORT] Saved embeddings to: {feats_path}")

index_df = df[["sample_path"]].copy()
index_path = os.path.join(OUT_EMB_DIR, "index.csv")
index_df.to_csv(index_path, index=False)
print(f"[EXPORT] Saved index to: {index_path}")

print("[EXPORT] Done.")
